# Kronos variant experiment: mini vs small vs base

**Question:** does scaling the Kronos time-series foundation model (4.1M → 24.7M → 102.3M params) buy directional accuracy worth the runtime/memory cost?

**Method:** walk-forward over the live watchlist (20 symbols × 25 most recent sessions each). For each as-of day: context = the 90 bars ending that day, 20 Monte-Carlo forecast samples (production settings: T=0.8, top_p=0.9, 3-day horizon), decision from the up-probability with the production HOLD band (0.4–0.6 unscored). Scored against the next session's close. **No lookahead** — pure price data, context strictly ≤ as-of.

**Pre-registered verdict criteria** (written before running):
- A variant must beat mini's hit rate by **≥ 2 SE (two-proportion z ≥ 2)** to justify switching.
- Ties on hit rate go to mini (cheapest); P&L is reported but is secondary (payoff asymmetry is noisier than hit rate at this n).

Runtime/memory (measured separately, CPU, production-equivalent 20-sample prediction): mini 0.6s/462MB · small 2.8s/462MB · base 5.3s/1,018MB peak.

In [1]:
import sys, time, resource
sys.path.insert(0, "/Users/UmarJahangir/Projects/quant-news")
import numpy as np, pandas as pd

SYMS = ["PANW", "BAC", "VZ", "HWM", "DOC", "HPQ", "LUV", "TPL", "MPWR", "MCD",
        "ROP", "ETR", "CMS", "XYZ", "HIG", "IP", "FLEX", "MET", "FIS", "TYL"]
DAYS, CONTEXT, SAMPLES = 25, 90, 20
HOLD_LO, HOLD_HI = 0.4, 0.6   # production band: inside = HOLD, unscored

In [2]:
# Price data from the app's own cache (writes to Postgres on the way through)
from services.cache_service import get_cache

frames = {}
for s in SYMS:
    df, _ = get_cache().get_stock_prices(s, "1y")
    df = df.rename(columns={"Open": "open", "High": "high", "Low": "low",
                            "Close": "close", "Volume": "volume"})
    df = df.dropna(subset=["open", "high", "low", "close"])
    if len(df) >= CONTEXT + DAYS + 2:
        frames[s] = df
print(f"{len(frames)} symbols with sufficient history")

20 symbols with sufficient history


In [3]:
from models.kronos.kronos import Kronos, KronosPredictor, KronosTokenizer

TOKENIZER = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")

def run_variant(size: str) -> pd.DataFrame:
    """Walk-forward one variant over every (symbol, day); returns per-call rows."""
    t_load = time.time()
    model = Kronos.from_pretrained(f"NeoQuasar/Kronos-{size}")
    pred = KronosPredictor(model, TOKENIZER, device="cpu")
    load_s = time.time() - t_load
    n_params = sum(p.numel() for p in model.parameters()) / 1e6

    rows, t0 = [], time.time()
    for sym, df in frames.items():
        n = len(df)
        for k in range(DAYS):
            end = n - 2 - k
            ctx = df.iloc[end - CONTEXT + 1: end + 1]
            x_ts = pd.DatetimeIndex(ctx.index)
            y_ts = pd.DatetimeIndex([x_ts[-1] + pd.Timedelta(days=i) for i in (1, 2, 3)])
            closes = []
            for _ in range(SAMPLES):
                out = pred.predict(ctx[["open", "high", "low", "close", "volume"]],
                                   x_ts, y_ts, pred_len=3, T=0.8, top_k=0,
                                   top_p=0.9, sample_count=1, verbose=False)
                closes.append(float(out["close"].iloc[0]))
            cur = float(ctx.close.iloc[-1])
            up_p = sum(1 for c in closes if c > cur) / SAMPLES
            actual = float(df.close.iloc[end + 1])
            rows.append({"symbol": sym, "as_of": str(ctx.index[-1])[:10],
                         "up_p": up_p, "move": actual / cur - 1})
    wall = time.time() - t0
    out = pd.DataFrame(rows)
    out.attrs.update(size=size, params_m=n_params, load_s=load_s, wall_s=wall)
    print(f"{size}: {n_params:.1f}M params · {len(out)} predictions · "
          f"{wall/60:.1f} min · {wall/len(out):.2f}s/pred")
    return out

def score(df: pd.DataFrame) -> dict:
    act = df[(df.up_p > HOLD_HI) | (df.up_p < HOLD_LO)].copy()
    act["dir"] = np.where(act.up_p > HOLD_HI, 1, -1)
    act["win"] = act.dir * act.move > 0
    act["pnl"] = act.dir * act.move * 1000
    n = len(act)
    hit = act.win.mean() if n else float("nan")
    return {"size": df.attrs["size"], "params_m": df.attrs["params_m"],
            "predictions": len(df), "active": n, "holds": len(df) - n,
            "hit": hit, "hit_se": (hit * (1 - hit) / n) ** 0.5 if n else None,
            "pnl_total": act.pnl.sum(), "pnl_per_trade": act.pnl.mean() if n else None,
            "sec_per_pred": df.attrs["wall_s"] / len(df)}

In [4]:
res_mini = run_variant("mini")

mini: 4.1M params · 500 predictions · 6.2 min · 0.75s/pred


In [5]:
res_small = run_variant("small")

small: 24.7M params · 500 predictions · 31.1 min · 3.74s/pred


In [6]:
res_base = run_variant("base")

base: 102.3M params · 500 predictions · 57.0 min · 6.84s/pred


In [7]:
results = [score(r) for r in (res_mini, res_small, res_base)]
tbl = pd.DataFrame(results).set_index("size")
display(tbl.round(4))

# Two-proportion z-tests vs mini (the pre-registered criterion)
def z_vs_mini(a, b):
    p = (a["hit"] * a["active"] + b["hit"] * b["active"]) / (a["active"] + b["active"])
    se = (p * (1 - p) * (1 / a["active"] + 1 / b["active"])) ** 0.5
    return (b["hit"] - a["hit"]) / se if se else 0.0

mini = results[0]
print("\n=== VERDICT (criterion: z >= 2 vs mini) ===")
for r in results[1:]:
    z = z_vs_mini(mini, r)
    delta = (r["hit"] - mini["hit"]) * 100
    verdict = "SWITCH JUSTIFIED" if z >= 2 else "keep mini"
    print(f"{r['size']:6s}: hit {r['hit']:.1%} vs mini {mini['hit']:.1%} "
          f"({delta:+.1f}pp, z={z:+.2f}) -> {verdict}")

,params_m,predictions,active,holds,hit,hit_se,pnl_total,pnl_per_trade,sec_per_pred
size,,,,,,,,,
mini,4.1080,500,308,192,0.5390,0.0284,444.3751,1.4428,0.7486
small,24.7414,500,341,159,0.5249,0.0270,472.8332,1.3866,3.7368
base,102.3106,500,342,158,0.5351,0.0270,946.4785,2.7675,6.8436



=== VERDICT (criterion: z >= 2 vs mini) ===
small : hit 52.5% vs mini 53.9% (-1.4pp, z=-0.36) -> keep mini
base  : hit 53.5% vs mini 53.9% (-0.4pp, z=-0.10) -> keep mini


## Interpretation notes

- **Hit rate is the decision metric.** P&L per trade is reported because Kronos's live value has come from payoff asymmetry (coin-flip hit, positive tail), but at n≈300-350 active calls per variant, P&L differences are dominated by a handful of large moves — do not switch variants on P&L alone.
- **HOLD counts matter**: a variant that abstains more but hits better on its active calls may be preferable — check `active` alongside `hit`.
- Cost side (measured separately): switching is one env var (`KRONOS_MODEL_SIZE`). small adds ~1-3 min to the daily 20-symbol run and nothing material to Railway cost; base additionally needs ≥1.5GB container headroom for its 1GB peak RSS.
- If a switch is justified, bump `PIPELINE_EPOCH` so live predictions are stamped with the variant change.